# Table Tennis Stroke Detection Pipeline

A comprehensive Jupyter notebook for extracting, processing, and training stroke detection models from table tennis videos.

**Pipeline Steps:**
1. Load annotations from CSV/XML
2. Extract motion features from videos (optical flow + frame difference)
3. Build feature table
4. Train Random Forest classifier

## 1. Setup and Imports

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import numpy as np
import pandas as pd
from lxml import etree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

try:
    import joblib
except ImportError:
    from sklearn.externals import joblib

print("All imports successful!")

## 2. Data Model: StrokeSegment

In [ ]:
@dataclass(frozen=True)
class StrokeSegment:
    """Represents a single stroke interval in a video."""
    video: str
    label: str
    start_frame: int
    end_frame: int

    @property
    def duration_frames(self) -> int:
        return max(0, self.end_frame - self.start_frame + 1)

# Test the class
example = StrokeSegment(video="test.mp4", label="forehand", start_frame=10, end_frame=50)
print(f"Example: {example}")
print(f"Duration: {example.duration_frames} frames")

## 3. Annotation Loading: Helper Functions

In [ ]:
_IDLE_LABELS = {"idle", "none", "background", "rest", "ready"}
_LABEL_ALIASES = {
    "serev": "serve",
}

def _norm_label(value: object) -> str:
    """Normalize label: lowercase and apply aliases (e.g., serev -> serve)."""
    label = str(value).strip().lower()
    return _LABEL_ALIASES.get(label, label)

def _norm_video_name(value: str) -> str:
    """Extract filename from video path."""
    p = Path(str(value).strip())
    return p.name

def _to_int(value: object) -> int:
    """Convert value to int via float (handles \"123.0\" style strings)."""
    return int(float(value))

# Test
print(_norm_label("SERVE"))
print(_norm_label("serev"))
print(_norm_video_name("/path/to/video.mp4"))

## 4. CSV Annotation Loading

In [ ]:
def _build_segments_from_frame_labels(df: pd.DataFrame) -> list[StrokeSegment]:
    """Convert frame-level labels to contiguous stroke segments."""
    segments: list[StrokeSegment] = []
    for video_name, group in df.groupby("video"):
        g = group.sort_values("frame")
        current_label = None
        start_frame = None
        prev_frame = None

        for row in g.itertuples(index=False):
            frame = _to_int(row.frame)
            label = str(row.label).strip().lower()
            if label in _IDLE_LABELS:
                label = "idle"

            if current_label is None:
                current_label = label
                start_frame = frame
                prev_frame = frame
                continue

            is_contiguous = frame == (prev_frame + 1)
            if label == current_label and is_contiguous:
                prev_frame = frame
                continue

            if current_label != "idle":
                segments.append(
                    StrokeSegment(
                        video=video_name,
                        label=current_label,
                        start_frame=start_frame,
                        end_frame=prev_frame,
                    )
                )
            current_label = label
            start_frame = frame
            prev_frame = frame

        if current_label is not None and current_label != "idle":
            segments.append(
                StrokeSegment(
                    video=video_name,
                    label=current_label,
                    start_frame=start_frame,
                    end_frame=prev_frame,
                )
            )
    return segments

print("Frame label builder ready")

In [ ]:
def _load_single_csv(csv_path: Path) -> list[StrokeSegment]:
    """Load stroke annotations from CSV."""
    df = pd.read_csv(csv_path)
    if df.empty:
        return []

    df.columns = [c.strip().lower() for c in df.columns]
    if "video" not in df.columns:
        if "video_path" in df.columns:
            df["video"] = df["video_path"]
        elif "filename" in df.columns:
            df["video"] = df["filename"]
        elif "video_name" in df.columns:
            df["video"] = df["video_name"]
        elif "stroke_id" in df.columns:
            df["video"] = csv_path.stem
        else:
            raise ValueError(f"{csv_path.name}: missing video/video_path/filename column")

    df["video"] = df["video"].map(_norm_video_name)

    if "stroke_type" in df.columns and "label" not in df.columns:
        df["label"] = df["stroke_type"]

    if {"start_frame", "end_frame", "label"}.issubset(df.columns):
        out: list[StrokeSegment] = []
        for row in df.itertuples(index=False):
            out.append(
                StrokeSegment(
                    video=row.video,
                    label=_norm_label(row.label),
                    start_frame=_to_int(row.start_frame),
                    end_frame=_to_int(row.end_frame),
                )
            )
        return out

    if {"frame", "label"}.issubset(df.columns):
        return _build_segments_from_frame_labels(df[["video", "frame", "label"]])

    if {"start_time", "end_time", "label", "fps"}.issubset(df.columns):
        out: list[StrokeSegment] = []
        for row in df.itertuples(index=False):
            fps = float(row.fps)
            out.append(
                StrokeSegment(
                    video=row.video,
                    label=_norm_label(row.label),
                    start_frame=_to_int(float(row.start_time) * fps),
                    end_frame=_to_int(float(row.end_time) * fps),
                )
            )
        return out

    raise ValueError(
        f"{csv_path.name}: unsupported CSV schema. Expected columns including either "
        "(video,start_frame,end_frame,label) or (video,frame,label) or "
        "(video,start_time,end_time,label,fps)."
    )

print("CSV loader ready")

## 5. XML Annotation Loading (CVAT Support)

In [ ]:
def _extract_video_from_cvat_meta(root: etree._Element, fallback_stem: str) -> str:
    """Extract video name from CVAT XML metadata."""
    task_name = root.findtext(".//meta//task//name")
    if task_name and task_name.strip():
        return _norm_video_name(task_name.strip())
    return fallback_stem

def _extract_label_from_cvat_track(track: etree._Element) -> str:
    """Extract the most common stroke_type from CVAT track boxes."""
    labels: list[str] = []

    for box in track.findall("box"):
        outside = box.attrib.get("outside", "0")
        if outside == "1":
            continue
        for attr in box.findall("attribute"):
            name = str(attr.attrib.get("name", "")).strip().lower()
            if name in {"stroke_type", "label", "class", "name"} and attr.text:
                labels.append(_norm_label(attr.text))

    if labels:
        return Counter(labels).most_common(1)[0][0]
    return _norm_label(track.attrib.get("label", "stroke"))

print("XML helper functions ready")

In [ ]:
def _extract_xml_segments(xml_path: Path) -> list[StrokeSegment]:
    """Load stroke annotations from XML (supports CVAT 1.1 and generic formats)."""
    tree = etree.parse(str(xml_path))
    root = tree.getroot()

    segments: list[StrokeSegment] = []

    # Handle CVAT 1.1 track schema first
    tracks = root.findall(".//track")
    if tracks:
        default_video = _extract_video_from_cvat_meta(root=root, fallback_stem=xml_path.stem)
        for track in tracks:
            boxes = track.findall("box")
            if not boxes:
                continue

            frames = []
            for box in boxes:
                outside = box.attrib.get("outside", "0")
                if outside == "1":
                    continue
                frame = box.attrib.get("frame")
                if frame is None:
                    continue
                frames.append(_to_int(frame))

            if not frames:
                continue

            segments.append(
                StrokeSegment(
                    video=_norm_video_name(default_video),
                    label=_extract_label_from_cvat_track(track),
                    start_frame=min(frames),
                    end_frame=max(frames),
                )
            )
        if segments:
            return segments

    # Fallback: generic XML parsing
    candidate_tags = ["stroke", "segment", "action", "event", "annotation"]

    nodes: list[etree._Element] = []
    for tag in candidate_tags:
        nodes.extend(root.findall(f".//{tag}"))

    for node in nodes:
        attrs = {k.lower(): v for k, v in node.attrib.items()}
        video = attrs.get("video") or attrs.get("video_name") or attrs.get("filename")
        label = attrs.get("label") or attrs.get("class") or attrs.get("name")

        start_frame = attrs.get("start_frame") or attrs.get("start") or attrs.get("begin")
        end_frame = attrs.get("end_frame") or attrs.get("end") or attrs.get("finish")

        if video is None:
            video_node = node.find("video") or node.find("filename")
            if video_node is not None and video_node.text:
                video = video_node.text

        if label is None:
            label_node = node.find("label") or node.find("class") or node.find("name")
            if label_node is not None and label_node.text:
                label = label_node.text

        if start_frame is None:
            sf_node = node.find("start_frame") or node.find("start") or node.find("begin")
            if sf_node is not None and sf_node.text:
                start_frame = sf_node.text

        if end_frame is None:
            ef_node = node.find("end_frame") or node.find("end") or node.find("finish")
            if ef_node is not None and ef_node.text:
                end_frame = ef_node.text

        if not all([video, label, start_frame, end_frame]):
            continue

        segments.append(
            StrokeSegment(
                video=_norm_video_name(video),
                label=_norm_label(label),
                start_frame=_to_int(start_frame),
                end_frame=_to_int(end_frame),
            )
        )

    return segments

print("XML loader ready")

In [ ]:
def load_annotations(csv_dir: Path, xml_dir: Path) -> list[StrokeSegment]:
    """Load all annotations from CSV and XML directories."""
    all_segments: list[StrokeSegment] = []

    for csv_path in sorted(csv_dir.glob("*.csv")):
        all_segments.extend(_load_single_csv(csv_path))

    for xml_path in sorted(xml_dir.glob("*.xml")):
        all_segments.extend(_extract_xml_segments(xml_path))

    dedup = {(s.video, s.label, s.start_frame, s.end_frame): s for s in all_segments}
    segments = list(dedup.values())
    segments.sort(key=lambda x: (x.video, x.start_frame, x.end_frame, x.label))
    return segments

def to_dataframe(segments: Iterable[StrokeSegment]) -> pd.DataFrame:
    """Convert stroke segments to DataFrame."""
    rows = [
        {
            "video": s.video,
            "label": s.label,
            "start_frame": s.start_frame,
            "end_frame": s.end_frame,
            "duration_frames": s.duration_frames,
        }
        for s in segments
    ]
    return pd.DataFrame(rows)

print("Annotation loading ready")

## 6. Video Feature Extraction

In [ ]:
def sample_frame_ids(start_frame: int, end_frame: int, n_frames: int) -> np.ndarray:
    """Sample n_frames uniformly from [start_frame, end_frame]."""
    start_frame = max(0, int(start_frame))
    end_frame = max(start_frame, int(end_frame))
    if n_frames <= 1:
        return np.array([start_frame], dtype=np.int32)
    return np.linspace(start_frame, end_frame, n_frames, dtype=np.int32)

# Test frame sampling
frames = sample_frame_ids(0, 100, 10)
print(f"Sample 10 frames from 0-100: {frames}")

In [ ]:
def extract_motion_features(
    video_path: Path,
    start_frame: int,
    end_frame: int,
    n_frames: int = 24,
) -> dict[str, float]:
    """Extract motion features (optical flow + frame difference) from video segment."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    frame_ids = sample_frame_ids(start_frame, end_frame, n_frames)
    gray_frames: list[np.ndarray] = []

    for fid in frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fid))
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_frames.append(gray)

    cap.release()

    if len(gray_frames) < 2:
        return {
            "flow_mag_mean": 0.0,
            "flow_mag_std": 0.0,
            "diff_mean": 0.0,
            "diff_std": 0.0,
            "duration_frames": float(max(0, end_frame - start_frame + 1)),
        }

    flow_mags = []
    diffs = []

    for i in range(1, len(gray_frames)):
        prev = gray_frames[i - 1]
        curr = gray_frames[i]

        flow = cv2.calcOpticalFlowFarneback(
            prev,
            curr,
            None,
            pyr_scale=0.5,
            levels=3,
            winsize=15,
            iterations=3,
            poly_n=5,
            poly_sigma=1.2,
            flags=0,
        )
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        flow_mags.append(float(np.mean(mag)))

        diff = cv2.absdiff(prev, curr)
        diffs.append(float(np.mean(diff)))

    return {
        "flow_mag_mean": float(np.mean(flow_mags)),
        "flow_mag_std": float(np.std(flow_mags)),
        "diff_mean": float(np.mean(diffs)),
        "diff_std": float(np.std(diffs)),
        "duration_frames": float(max(0, end_frame - start_frame + 1)),
    }

print("Feature extraction ready")

## 7. Dataset Building

In [ ]:
def _resolve_video_path(videos_dir: Path, video_token: str) -> Path | None:
    """Resolve video file by name, stem, or case-insensitive match."""
    direct = videos_dir / video_token
    if direct.exists():
        return direct

    stem = Path(video_token).stem
    matches = sorted(videos_dir.glob(f"{stem}.*"))
    if matches:
        return matches[0]

    lower_stem = stem.lower()
    for p in videos_dir.iterdir():
        if p.is_file() and p.stem.lower() == lower_stem:
            return p
    return None

print("Video path resolver ready")

In [ ]:
def build_feature_table(
    videos_dir: Path,
    csv_dir: Path,
    xml_dir: Path,
    output_csv: Path,
    n_frames: int = 24,
) -> pd.DataFrame:
    """Build feature table from videos and annotations."""
    segments = load_annotations(csv_dir=csv_dir, xml_dir=xml_dir)
    rows = []

    for seg in tqdm(segments, desc="Extracting segment features"):
        video_path = _resolve_video_path(videos_dir=videos_dir, video_token=seg.video)
        if video_path is None:
            print(f"  Warning: Video not found for {seg.video}, skipping")
            continue

        try:
            features = extract_motion_features(
                video_path=video_path,
                start_frame=seg.start_frame,
                end_frame=seg.end_frame,
                n_frames=n_frames,
            )
            row = {
                "video": video_path.name,
                "label": seg.label,
                "start_frame": seg.start_frame,
                "end_frame": seg.end_frame,
            }
            row.update(features)
            rows.append(row)
        except Exception as e:
            print(f"  Error processing {seg.video}: {e}")
            continue

    df = pd.DataFrame(rows)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Saved {len(df)} features to {output_csv}")
    return df

print("Feature table builder ready")

## 8. Model Training

In [ ]:
FEATURE_COLUMNS = [
    "flow_mag_mean",
    "flow_mag_std",
    "diff_mean",
    "diff_std",
    "duration_frames",
]

def train_model(
    features_csv: Path,
    model_out: Path,
    report_out: Path,
    random_state: int = 42,
) -> dict:
    """Train Random Forest classifier on extracted features."""
    df = pd.read_csv(features_csv)
    if df.empty:
        raise ValueError("Feature CSV is empty. Build dataset first.")

    missing = [c for c in FEATURE_COLUMNS + ["label"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    X = df[FEATURE_COLUMNS]
    y = df["label"]

    if len(y.unique()) < 2:
        print(f"Warning: Only {len(y.unique())} class(es) in data. Training may be unstable.")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=random_state,
        stratify=y,
    )

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    random_state=random_state,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    report = classification_report(y_test, preds, output_dict=True, zero_division=0)

    model_out.parent.mkdir(parents=True, exist_ok=True)
    report_out.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_out)
    with report_out.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print(f"Saved model to {model_out}")
    print(f"Saved report to {report_out}")
    return report

print("Model trainer ready")

## 9. Main Pipeline Execution

### Configure Paths

Update these paths to match your data structure:

In [ ]:
# For Kaggle
BASE_INPUT = Path("/kaggle/input/stroke-data")
BASE_OUTPUT = Path("/kaggle/output")

# For local testing (uncomment to use)
# BASE_INPUT = Path("./data")
# BASE_OUTPUT = Path("./outputs")

# Setup paths
videos_dir = BASE_INPUT / "raw_videos"
csv_dir = BASE_INPUT / "annotations" / "csv"
xml_dir = BASE_INPUT / "annotations" / "xml"

features_csv = BASE_OUTPUT / "stroke_features.csv"
model_out = BASE_OUTPUT / "stroke_classifier.joblib"
report_out = BASE_OUTPUT / "classification_report.json"

print(f"Input directories:")
print(f"  Videos: {videos_dir}")
print(f"  CSV annotations: {csv_dir}")
print(f"  XML annotations: {xml_dir}")
print(f"\nOutput files:")
print(f"  Features: {features_csv}")
print(f"  Model: {model_out}")
print(f"  Report: {report_out}")

### Step 1: Build Feature Table

In [ ]:
# Ensure directories exist
for d in [videos_dir, csv_dir, xml_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Build feature table
print("Building feature table...\n")
df_features = build_feature_table(
    videos_dir=videos_dir,
    csv_dir=csv_dir,
    xml_dir=xml_dir,
    output_csv=features_csv,
    n_frames=24,
)

print(f"\nExtracted {len(df_features)} stroke features")
if len(df_features) > 0:
    print(f"Classes: {df_features['label'].unique()}")
    print(f"\nFeature statistics:\n{df_features[FEATURE_COLUMNS].describe()}")
else:
    print("No features extracted. Check annotation and video files.")

### Step 2: Train Model (if sufficient data)

In [ ]:
if len(df_features) >= 20 and df_features["label"].nunique() >= 2:
    print("Training Random Forest classifier...\n")
    report = train_model(features_csv, model_out, report_out)
    
    macro_f1 = report.get("macro avg", {}).get("f1-score", 0.0)
    print(f"\nMacro F1-Score: {macro_f1:.4f}")
else:
    min_samples = max(20, df_features["label"].nunique() * 2) if len(df_features) > 0 else 20
    print(f"Skipping training: need at least {min_samples} samples and 2+ classes")
    print(f"Have {len(df_features)} samples, {df_features['label'].nunique() if len(df_features) > 0 else 0} classes")

## Summary

This notebook provides:

✅ **Robust annotation loading** - supports CSV and XML (including CVAT format)  
✅ **Motion feature extraction** - optical flow + frame difference  
✅ **Automated dataset building** - from raw videos and annotations  
✅ **Model training** - Random Forest classifier with evaluation  

**Next steps:**
1. Upload data in expected directory structure
2. Run cells in order
3. Check `/kaggle/output/` for results: features.csv, model.joblib, classification_report.json